# 🛡️ Transaction Fraud Scorer Agentic AI (Google Colab Setup & Run)

Welcome to the **Transaction Fraud Scorer Agentic AI** interactive Colab notebook!

This notebook demonstrates how to:
1. Clone the repository, install dependencies, and initialize the **SQLite RDBMS Database** (`src/sqllite/fraud_scorer.db`) from `docs/rdbms-model-schema/schema.sql`.
2. Launch the **FastAPI Gateway** (`src/api_gateway`) exposed via **Cloudflare Tunneling (`cloudflared`)**.
3. Verify that incoming `POST /api/v1/transactions/score` requests are returned in the **WebSocket response format** (`GET /ws/transactions/events`) with `score = 100.0` and `status = "APPROVE"`.
4. (Optional) Install and run a local **Ollama** server (`llama3.2:3b`) and **Model Context Protocol (MCP)** fraud scoring tools.

## Step 1: Verify Hardware Environment
Check if a GPU is assigned.

In [2]:
!nvidia-smi || echo "Running in CPU mode. Lightweight models like llama3.2:3b run well on CPU too!"

/bin/bash: line 1: nvidia-smi: command not found
Running in CPU mode. Lightweight models like llama3.2:3b run well on CPU too!


## Step 2: Clone Repository & Install Dependencies
Clone the project repository and install required packages (`fastapi`, `uvicorn`, `websockets`, `langchain`, `langchain-ollama`, `mcp`, `pydantic`, `rich`).

In [ ]:
import os
import sys

REPO_URL = "https://github.com/daknamsubho/TransactionFraudScorerAgenticAI.git"
REPO_DIR = "/content/TransactionFraudScorerAgenticAI" if os.path.exists("/content") else os.path.abspath("TransactionFraudScorerAgenticAI")
BRANCH = "feature/local_models"

# Clone or update repository on the target branch
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"📥 Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Repository found at {REPO_DIR}. Fetching latest changes on {BRANCH}...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
print("📂 Working directory:", os.getcwd())

# Install project dependencies
!pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "websockets>=12.0"
!pip install -q "langchain>=1.4.0" "langchain-ollama>=1.1.0" "mcp>=1.3.0"
!pip install -q "pydantic>=2.7.0" "pydantic-settings>=2.2.0" "rich>=13.7.0" "httpx>=0.27.0"
!pip install -q -e .

src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("✅ Environment and dependencies configured successfully!")

## Step 2.5: Install & Initialize SQLite RDBMS (`src/sqllite`) + Verify CRUD Queries
Read the SQLite installation script (`src/sqllite/install_sqlite.sh` / `install_and_setup_sqlite`) and DDL schema (`docs/rdbms-model-schema/schema.sql`) to initialize `src/sqllite/fraud_scorer.db`, then verify `INSERT`, `SELECT`, `UPDATE`, and `DELETE` queries across `user_profiles`, `transactions`, and `transaction_scores`.

In [ ]:
!bash src/sqllite/install_sqlite.sh

from sqllite import (
    calculate_user_profile_metrics,
    delete_transaction,
    install_and_setup_sqlite,
    insert_transaction,
    insert_transaction_score,
    insert_user_profile,
    select_transaction_with_score,
    select_user_profile,
    update_transaction_decision,
)

db_info = install_and_setup_sqlite()
print("📦 SQLite Setup Info:", db_info)

# 1. INSERT UserProfile (username, created_date, address)
profile = insert_user_profile(
    username="jdoe_99",
    created_date="2025-01-15T10:00:00+00:00",
    address="123 Market St, Suite 400",
    city="San Francisco",
    state="CA",
    country="US",
    postal_code="94103",
)
print("\n👤 [SELECT UserProfile]:", select_user_profile("jdoe_99"))

# 2. INSERT Transaction & Score
insert_transaction(
    transaction_id="tx_sqlite_demo_001",
    username="jdoe_99",
    transaction_type="CARD",
    amount=275.50,
    latitude=37.7749,
    longitude=-122.4194,
    city="San Francisco",
    state="CA",
    country="US",
    merchant_id="merch_88412",
    merchant_name="ElectroWorld Online",
    mcc="5732",
    payment_method_type="credit_card",
    card_or_account_last_four="4242",
    is_card_present=False,
    status="REVIEW",
)
insert_transaction_score(
    transaction_id="tx_sqlite_demo_001",
    score=48.5,
    approval_status="REVIEW",
    reasoning_summary="Moderate risk online electronics purchase.",
    tools_invoked=["check_velocity", "check_geo_anomaly", "check_merchant_risk", "lookup_account_history"],
)

# 3. UPDATE Decision & SELECT joined Transaction + Score
updated = update_transaction_decision(
    transaction_id="tx_sqlite_demo_001",
    accepted=True,
    reason="Verified cardholder via 2FA challenge.",
)
print("\n💳 [SELECT Transaction + Score after UPDATE]:", select_transaction_with_score("tx_sqlite_demo_001"))

# 4. Dynamic Tool Calculation (account_age_days, prior_disputes, avg_transaction_amount, trust_tier)
print("\n📊 [Dynamic UserProfile Metrics]:", calculate_user_profile_metrics("jdoe_99"))

# 5. DELETE demo transaction
deleted = delete_transaction("tx_sqlite_demo_001")
print("\n🗑️ [DELETE Demo Transaction]:", deleted)

## Step 2.6: Seed the SQLite Database (`docs/dataset`)
Seed the SQLite RDBMS (`src/sqllite/fraud_scorer.db`) from `docs/dataset/` with:
- **30 good personas** created in the last 1 year (`good_persona_01`..`good_persona_30`)
- **10 bad personas** created in the last 5 months (`bad_persona_01`..`bad_persona_10`)
- **20 good personas** with 10–30 good transactions each week for the last 7 months (30 weeks)
- **5 bad personas** with 2–5 good transactions in the last 2 weeks

In [ ]:
from sqllite import calculate_user_profile_metrics, seed_database

seed_summary = seed_database()
print("🌱 SQLite Seed Summary:", seed_summary)
print("👤 Sample Good Persona Metrics (good_persona_01):", calculate_user_profile_metrics("good_persona_01"))
print("👤 Sample Bad Persona Metrics (bad_persona_01):", calculate_user_profile_metrics("bad_persona_01"))

## Step 3: Start FastAPI Gateway & Expose via Cloudflare Tunnel (`cloudflared`)
Start the FastAPI API Gateway (`src/api_gateway`) on port `8000` in the background and expose it externally via a **Cloudflare Quick Tunnel** (`https://*.trycloudflare.com`).

Copy the printed URLs into your local website client to ingest transactions and stream real-time WebSocket score events!

In [ ]:
import threading
import time
import httpx
import uvicorn

from api_gateway.app import app
from api_gateway.tunnel import start_cloudflare_tunnel

GATEWAY_PORT = 8000

def _run_uvicorn() -> None:
    uvicorn.run(app, host="0.0.0.0", port=GATEWAY_PORT, log_level="info")

# Start FastAPI server in a daemon thread if not already listening
try:
    httpx.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=1.0)
    print("✅ FastAPI Gateway is already running on port", GATEWAY_PORT)
except Exception:
    server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("✅ FastAPI Gateway started on http://127.0.0.1:8000")

# Start Cloudflare Quick Tunnel
tunnel_proc, public_url = start_cloudflare_tunnel(port=GATEWAY_PORT)
ws_url = public_url.replace("https://", "wss://") + "/ws/transactions/events"

print("\n" + "=" * 72)
print("🌐 CLOUDFLARE TUNNEL ACTIVE — USE THESE ENDPOINTS IN YOUR LOCAL WEBSITE")
print("=" * 72)
print(f"🔗 Public Base URL : {public_url}")
print(f"📖 Swagger UI Docs : {public_url}/docs")
print(f"📨 Score Endpoint  : POST {public_url}/api/v1/transactions/score")
print(f"⚖️ Decision API    : POST {public_url}/api/v1/transactions/decision")
print(f"📡 WebSocket Stream: {ws_url}")
print("-" * 72)
print("🖥️  To launch the local WebUI on your computer, run:")
print("    uv run fraud-web-client --port 3000")
print(f"    Then open http://127.0.0.1:3000 and paste {public_url} into 'API Gateway Base URL'.")
print("=" * 72)

## Step 4: Verify API Gateway Returns Score Request via WebSocket (`Score=100.0`, `Status=APPROVE`)
Test that:
1. Sending a `POST /api/v1/transactions/score` request immediately returns `score: 100.0` and `status: "APPROVE"`.
2. Connected WebSocket clients on `/ws/transactions/events` receive the `WebSocketTransactionEvent` containing `transaction_id`, `username`, `amount`, `score: 100.0`, and `score_details` (including the echoed `request_payload`).
3. `POST /api/v1/transactions/decision` records an `Accept`/`Reject` decision with a `Reason`.

In [ ]:
import asyncio
import json
import httpx
import websockets

local_ws_url = f"ws://127.0.0.1:{GATEWAY_PORT}/ws/transactions/events"
local_http_url = f"http://127.0.0.1:{GATEWAY_PORT}"

sample_card_payload = {
    "username": "jdoe_99",
    "datetime": "2026-10-31T18:09:11Z",
    "ip": "198.51.100.42",
    "device_id": "dev_macbook_pro_91a",
    "location": {
        "latitude": 37.7749,
        "longitude": -122.4194,
        "city": "San Francisco",
        "state": "CA",
        "country": "US",
        "postal_code": "94103"
    },
    "transaction_details": {
        "transaction_type": "CARD",
        "amount": 1450.00,
        "currency": "USD",
        "merchant": {
            "merchant_id": "merch_88412",
            "name": "ElectroWorld Online",
            "mcc": "5732",
            "country": "US"
        },
        "card_details": {
            "card_type": "CREDIT",
            "card_network": "VISA",
            "card_last_four": "4242",
            "is_card_present": False,
            "expiration_month": 8,
            "expiration_year": 2028,
            "cvv_verified": True,
            "avs_result": "Y"
        }
    }
}

async with websockets.connect(local_ws_url) as ws:
    async with httpx.AsyncClient(timeout=30.0) as client:
        # 1. Score the transaction (returns score=100.0, status=APPROVE)
        score_resp = await client.post(f"{local_http_url}/api/v1/transactions/score", json=sample_card_payload)
        score_data = score_resp.json()
        print("✅ POST /api/v1/transactions/score Response:")
        print(json.dumps(score_data, indent=2))

        # 2. Receive real-time WebSocket event
        ws_msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=5.0))
        print("\n📡 WebSocket Event Received (/ws/transactions/events):")
        print(json.dumps(ws_msg, indent=2))

        # 3. Submit Accept/Reject decision
        decision_payload = {
            "transaction_id": score_data["transaction_id"],
            "accept": True,
            "reason": "Customer verified transaction via 2FA challenge."
        }
        decision_resp = await client.post(f"{local_http_url}/api/v1/transactions/decision", json=decision_payload)
        print("\n⚖️ POST /api/v1/transactions/decision Response:")
        print(json.dumps(decision_resp.json(), indent=2))

## Step 4.5: Verify OpenAPI RDBMS Endpoints (`GET`, `POST`, `PUT`, `DELETE`) for Paginated WebClient Pages
Test the FastAPI Gateway's RDBMS CRUD and paginated `GET` endpoints used by the WebClient pages (`users.html`, `transactions.html`, `scores.html`):
- `/api/v1/db/user-profiles`
- `/api/v1/db/transactions`
- `/api/v1/db/transaction-scores`

In [ ]:
async with httpx.AsyncClient(timeout=30.0) as client:
    # 1. Paginated GET User Profiles
    users_page = (await client.get(f"{local_http_url}/api/v1/db/user-profiles?page=1&page_size=5")).json()
    print("👤 Paginated GET /api/v1/db/user-profiles:")
    print(json.dumps(users_page, indent=2))

    # 2. PUT Update User Profile
    updated_user = (
        await client.put(
            f"{local_http_url}/api/v1/db/user-profiles/jdoe_99",
            json={"address": "500 Howard St", "city": "San Francisco", "state": "CA", "country": "US", "postal_code": "94105"},
        )
    ).json()
    print("\n✏️ PUT /api/v1/db/user-profiles/jdoe_99:")
    print(json.dumps(updated_user, indent=2))

    # 3. Paginated GET Transactions
    tx_page = (await client.get(f"{local_http_url}/api/v1/db/transactions?page=1&page_size=5")).json()
    print("\n💳 Paginated GET /api/v1/db/transactions:")
    print(json.dumps(tx_page, indent=2))

    # 4. Paginated GET Transaction Scores
    scores_page = (await client.get(f"{local_http_url}/api/v1/db/transaction-scores?page=1&page_size=5")).json()
    print("\n📊 Paginated GET /api/v1/db/transaction-scores:")
    print(json.dumps(scores_page, indent=2))

## Step 5: (Optional) Install Ollama in Colab & Launch Background Server
We install `zstd` (required by the Ollama installer) and Ollama using its official one-line script, then start the background daemon process on `http://localhost:11434`.

In [ ]:
import subprocess
import time

import requests

print("🔧 Installing prerequisite zstd package...")
!apt-get update -qq && apt-get install -y -qq zstd

print("📥 Downloading and installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Starting Ollama daemon in background...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Check health
for i in range(10):
    try:
        resp = requests.get("http://localhost:11434/api/version")
        if resp.status_code == 200:
            print(f"✅ Ollama server is UP and ready! Version: {resp.json().get('version')}")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Ollama is still initializing. Please wait a moment.")

## Step 6: (Optional) Pull Open-Source Model (`llama3.2:3b`)
Download `llama3.2:3b` (~2.0 GB). It provides high reasoning quality and native tool calling.

In [ ]:
print("📦 Pulling llama3.2:3b model into Ollama...")
!ollama pull llama3.2:3b

print("\n📋 Available local models:")
!ollama list

## Step 7: Test FastMCP Server, Modular Risk Tools & LangChain Configuration
Verify that:
1. Each MCP tool is loaded from its own dedicated module (`check_velocity.py`, `check_geo_anomaly.py`, `check_merchant_risk.py`, `lookup_account_history.py`).
2. The **FastMCP Server** (`create_server()`) registers and exposes all 4 tools (`list_tools()` and `call_tool()`).
3. All tools are configured as **LangChain `@tool` adapters** (`get_langchain_fraud_tools()` & `configure_langchain_with_mcp_tools()`).

In [ ]:
from langchain_ollama import ChatOllama

from fraud_scorer.agent.tools import configure_langchain_with_mcp_tools, get_langchain_fraud_tools
from fraud_scorer.mcp.server import create_server
from fraud_scorer.mcp.tools.check_geo_anomaly import check_geo_anomaly
from fraud_scorer.mcp.tools.check_merchant_risk import check_merchant_risk
from fraud_scorer.mcp.tools.check_velocity import check_velocity
from fraud_scorer.mcp.tools.lookup_account_history import lookup_account_history

# 1. Inspect FastMCP Server & Registered Tools
fastmcp_server = create_server()
mcp_tools = await fastmcp_server.list_tools()
print(f"✅ FastMCP Server '{fastmcp_server.name}' initialized with {len(mcp_tools)} tools:")
for mt in mcp_tools:
    print(f"  • [FastMCP] {mt.name}")

# 2. Test individual modular tool files connected to SQLite RDBMS (seeded personas)
print("\n🔍 Testing MCP tools connected to SQLite RDBMS (good_persona_01 & bad_persona_01):")
print("  • check_velocity (good_persona_01)        :", check_velocity("good_persona_01", 60))
print("  • check_geo_anomaly (good_persona_01)     :", check_geo_anomaly("good_persona_01", 37.7749, -122.4194))
print("  • check_merchant_risk (merch_grocery_01)  :", check_merchant_risk("merch_grocery_01", "5411"))
print("  • lookup_account_history (good_persona_01):", lookup_account_history("good_persona_01"))
print("  • lookup_account_history (bad_persona_01) :", lookup_account_history("bad_persona_01"))

# 3. Configure with LangChain
lc_tools = get_langchain_fraud_tools()
llm = ChatOllama(base_url="http://localhost:11434", model="llama3.2:3b", temperature=0.0)
llm_with_mcp_tools = configure_langchain_with_mcp_tools(llm)
print(f"\n🔗 Configured {len(lc_tools)} LangChain MCP Tool Adapters with ChatOllama:")
for t in lc_tools:
    summary = t.description.strip().splitlines()[0]
    print(f"  • {t.name}: {summary}")

## Step 8: Initialize LangChain Agent Orchestrator with `ChatOllama`
Initialize `FraudScorerOrchestrator` pointing to the live Ollama daemon.

In [ ]:
from fraud_scorer.agent.orchestrator import FraudScorerOrchestrator, display_assessment, is_ollama_available
from fraud_scorer.models.transaction import GeoLocation, MerchantInfo, PaymentMethod, Transaction

print("Ollama server connected:", is_ollama_available("http://localhost:11434"))

# Initialize Orchestrator backed by ChatOllama
orchestrator = FraudScorerOrchestrator(
    ollama_base_url="http://localhost:11434", model_name="llama3.2:3b", provider="ollama"
)
print(f"✅ Fraud Scorer Orchestrator initialized using {orchestrator.model_name}!")

## Step 9: Scenario 1 - Normal Local Coffee Purchase
Evaluate a legitimate transaction ($42.50 in San Francisco at Blue Bottle Coffee by an established account).
Expected outcome: **`APPROVE`** with low composite risk score.

In [ ]:
tx_normal = Transaction(
    account_id="acc_normal_01",
    amount=42.50,
    currency="USD",
    location=GeoLocation(latitude=37.7749, longitude=-122.4194, city="San Francisco", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="4242", is_card_present=True),
    merchant=MerchantInfo(merchant_id="merch_cafe_88", name="Blue Bottle Coffee", mcc="5499"),
)

print("🔍 Evaluating Scenario 1 with LangChain & Ollama...")
assessment_1 = await orchestrator.evaluate_transaction(tx_normal)
display_assessment(tx_normal, assessment_1)

## Step 10: Scenario 2 - Multi-Vector Fraudulent Transaction
Evaluate a high-risk transaction ($3,500.00 in Tokyo at a casino shortly after London activity with high velocity and new untrusted account profile).
Expected outcome: **`DECLINE`** with triggered critical risk factors (impossible travel, gambling MCC, burst velocity).

In [ ]:
tx_suspicious = Transaction(
    account_id="acc_suspicious_01",
    amount=3500.00,
    currency="USD",
    location=GeoLocation(latitude=35.6762, longitude=139.6503, city="Tokyo", country="JP"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="9911", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_casino_77", name="Tokyo High Roller Casino", mcc="7995"),
)

print("🔍 Evaluating Scenario 2 with LangChain & Ollama...")
assessment_2 = await orchestrator.evaluate_transaction(tx_suspicious)
display_assessment(tx_suspicious, assessment_2)